# Module 2.1: From Similarity Search to Connected Context

Compare four retrieval patterns over the same hotel graph: vector, hybrid, Vector-Cypher, and Text2Cypher. Each pattern works best for a different query shape.

The notebook retrieves and displays evidence directly for comparison. Answer generation is outside this exercise.

## Before you run this notebook

From `notebooks/02-connected-context/`, run the Module 2 preparation command. It creates the deterministic 30-document sample when needed, reuses the Nova embeddings in `:Chunk.embedding`, and provisions both retrieval indexes:

```bash
uv run prepare_graph.py --mode lite
```

The command is idempotent. If the graph and indexes are ready, it reports their status and leaves them unchanged.

In [1]:
import sys, os
# Add notebooks/ root to path so the shared `workshop` package is importable
_notebooks_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _notebooks_root not in sys.path:
    sys.path.insert(0, _notebooks_root)


In [2]:
import os

from dotenv import load_dotenv, find_dotenv
from IPython.display import HTML, Markdown, display
from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import (
    HybridRetriever,
    Text2CypherRetriever,
    VectorCypherRetriever,
    VectorRetriever,
)
from neo4j_graphrag.types import RetrieverResultItem

load_dotenv(find_dotenv())

from workshop.aws_region import aws_region, configure_aws_region
from workshop.bedrock_providers import BedrockEmbeddings, BedrockLLM
from workshop.graph_connection import neo4j_auth, neo4j_uri, require_neo4j_env
from workshop.graph_schema import GRAPH_SCHEMA
from workshop.retrieval_contract import CHUNK_FULLTEXT_INDEX, CHUNK_VECTOR_INDEX
from workshop.retrieval_setup import fixture_problems, verify_retrieval_indexes

configure_aws_region()
require_neo4j_env()
driver = GraphDatabase.driver(neo4j_uri(), auth=neo4j_auth())
driver.verify_connectivity()
print('Connected to Neo4j.')

Connected to Neo4j.


## Verify the prepared graph

Verify the graph before running retrieval. This check confirms that both indexes are online and use the expected label, property, dimensions, and similarity function. It also checks every graph fixture required by the demonstrations.

In [3]:
try:
    verify_retrieval_indexes(driver)
    problems = fixture_problems(driver)
    if problems:
        raise RuntimeError('; '.join(problems))
except Exception as exc:
    raise RuntimeError(
        f'Module 2.1 is not ready: {exc}\n'
        'From notebooks/02-connected-context, run: uv run prepare_graph.py --mode lite'
    ) from exc

print(f'✓ {CHUNK_VECTOR_INDEX} is online')
print(f'✓ {CHUNK_FULLTEXT_INDEX} is online')
print('✓ Demo-critical fixtures are present')

✓ hotel_chunk_embeddings is online
✓ hotel_chunk_fulltext is online
✓ Demo-critical fixtures are present


## The pinned hotel graph schema

The extraction pipeline uses one explicit schema. Review its relationships before running the traversal-based retrievers that use them.

In [4]:
pattern_rows = ''.join(
    f'<tr><td><strong>{source}</strong></td><td>-[:{relationship}]-&gt;</td>'
    f'<td><strong>{target}</strong></td></tr>'
    for source, relationship, target in GRAPH_SCHEMA['patterns']
)
display(HTML(
    '<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>'
    f'</thead><tbody>{pattern_rows}</tbody></table>'
    '<p>Each extracted entity also points to its source '    '<code>(entity)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>'
))

From,Relationship,To
Hotel,-[:HAS_ROOM]->,Room
Hotel,-[:OFFERS_AMENITY]->,Amenity
Hotel,-[:HAS_POLICY]->,Policy
Hotel,-[:PROVIDES_SERVICE]->,Service


## Shared query embedding contract

The graph build wrote 1024-dimensional Amazon Nova 2 embeddings with the `GENERIC_INDEX` purpose. Retrieval uses the same provider contract to embed each query. The stored chunk embeddings remain unchanged.

In [5]:
embedder = BedrockEmbeddings(region_name=aws_region())

def show_results(question, result, why):
    print(f'Question: {question}\n')
    for number, item in enumerate(result.items, 1):
        score = (item.metadata or {}).get('score')
        score_text = 'n/a' if score is None else f'{score:.4f}'
        content = str(item.content)
        preview = content[:700] + ('…' if len(content) > 700 else '')
        print(f'[{number}] score={score_text}\n{preview}\n')
    print(f'Pattern fit: {why}')

## Pattern 1: Vector retrieval for semantic lookup

Use vector retrieval when the question may paraphrase the source and a relevant chunk contains the answer.

In [6]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    embedder=embedder,
    return_properties=['text'],
)
vector_question = (
    'What is the standard check-in time at the AnyCompany Cairo Nile View hotel?'
)
vector_result = vector_retriever.search(
    query_text=vector_question,
    top_k=3,
)
show_results(
    vector_question,
    vector_result,
    'Semantic similarity finds the policy wording even when the question is paraphrased.',
)

Question: What is the standard check-in time at the AnyCompany Cairo Nile View hotel?

[1] score=0.9070
{'text': "# AnyCompany Cairo Nile View\n\n## Hotel Overview\nAnyCompany Cairo Nile View is located in Cairo, Egypt. Pharaonic luxury overlooking the eternal Nile\n\n**Guest Rating:** 4.5/5.0\n**Total Rooms:** 250\n\n## Contact Information\n**Address:** 789 Corniche el-Nil, Cairo 11519\n**Phone:** +20-2-555-0001\n**Email:** cairo@anycompany.com\n\n## Check-in & Check-out\n\n### Check-in Time\nStandard check-in time is 3:00 PM. Early check-in is subject to availability and may be requested at the time of booking or by contacting the hotel directly.\n\n### Check-out Time\nStandard check-out time is 11:00 AM. Late check-out until 2:00 PM may be available for an additional fee of $50, subject to …

[2] score=0.8786
{'text': "# AnyCompany Giza Pyramids\n\n## Hotel Overview\nAnyCompany Giza Pyramids is located in Cairo, Egypt. Ancient wonders meet modern comfort\n\n**Guest Rating:** 4.6/5.0

## Pattern 2: Hybrid retrieval for exact names and identifiers

Embeddings can blur exact identifiers. In the deterministic lite sample, the chunk for Windward Mile Tower contains postal code `60611`. Historical validation ranked that chunk 12th with pure vector search. Compare the top five vector results with hybrid retrieval. The hybrid query uses the full question for its vector signal and the postal code for its full-text signal, then combines the rankings.

In [7]:
identifier_question = 'What is the cancellation policy for the hotel at 60611?'
vector_identifier_result = vector_retriever.search(
    query_text=identifier_question,
    top_k=5,
)

hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=CHUNK_VECTOR_INDEX,
    fulltext_index_name=CHUNK_FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=['text'],
)
hybrid_result = hybrid_retriever.search(
    query_text='60611',
    query_vector=vector_identifier_result.metadata['query_vector'],
    top_k=5,
    ranker='linear',
    alpha=0.2,
)

show_results(
    identifier_question,
    vector_identifier_result,
    'This semantic-only search gives exact postal codes a weak signal.',
)
print('\n' + '=' * 80 + '\n')
show_results(
    identifier_question,
    hybrid_result,
    'Full-text matching preserves 60611 while the separately supplied question vector handles the policy wording.',
)
assert any('60611' in str(item.content) for item in hybrid_result.items)

Question: What is the cancellation policy for the hotel at 60611?

[1] score=0.8279
{'text': "# Melody Lane Hotel\n\n## Hotel Overview\nMelody Lane Hotel is located in Austin, TX, USA. Stylish hotel in a prime location\n\n**Guest Rating:** 4.5/5.0\n**Total Rooms:** 320\n\n## Contact Information\n**Address:** 1071 Congress Avenue, Austin, TX 78701\n**Phone:** +1-512-555-0131\n**Email:** austin1@anycompany.com\n\n## Check-in & Check-out\n\n### Check-in Time\nStandard check-in time is 3:00 PM. Early check-in is subject to availability and may be requested at the time of booking or by contacting the hotel directly.\n\n### Check-out Time\nStandard check-out time is 11:00 AM. Late check-out until 2:00 PM may be available for an additional fee of $50, subject to availability.\n\n#…

[2] score=0.8249
{'text': "# Cliffside Coast Resort\n\n## Hotel Overview\nCliffside Coast Resort is located in Kauai, HI, USA. Garden island luxury with natural beauty\n\n**Guest Rating:** 4.7/5.0\n**Total Rooms:*

## Pattern 3: Vector-Cypher for graph-enriched lookup

Use Vector-Cypher to find a relevant chunk by semantic similarity, then add structured hotel and amenity context through graph traversal.

In [8]:
retrieval_query = '''
MATCH (hotel:Hotel)-[:FROM_CHUNK]->(node)
OPTIONAL MATCH (hotel)-[relationship]->(detail)
WHERE type(relationship) IN [
    'HAS_ROOM', 'OFFERS_AMENITY', 'HAS_POLICY', 'PROVIDES_SERVICE'
]
WITH node, score, hotel,
     collect(DISTINCT {
         relationship: type(relationship),
         name: coalesce(detail.name, detail.type),
         description: detail.description
     })[..12] AS related
RETURN node.text AS chunk, score,
       hotel { .name, .address, .guest_rating } AS hotel,
       related
'''

def graph_result_formatter(record):
    content = {
        'chunk': record.get('chunk'),
        'hotel': record.get('hotel'),
        'related': record.get('related'),
    }
    return RetrieverResultItem(
        content=str(content),
        metadata={'score': record.get('score')},
    )

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    retrieval_query=retrieval_query,
    embedder=embedder,
    result_formatter=graph_result_formatter,
)
graph_question = (
    'Tell me about the hotel at 789 Avenue des Champs-Élysées and its amenities.'
)
graph_result = vector_cypher_retriever.search(
    query_text=graph_question,
    top_k=2,
)
show_results(
    graph_question,
    graph_result,
    'Vector search locates source chunks; Cypher adds connected hotel entities and typed relationships.',
)

Question: Tell me about the hotel at 789 Avenue des Champs-Élysées and its amenities.

[1] score=0.8731
{'chunk': "# AnyCompany Paris Champs-Élysées\n\n## Hotel Overview\nAnyCompany Paris Champs-Élysées is located in Paris, France. Luxury on the world's most famous avenue\n\n**Guest Rating:** 4.9/5.0\n**Total Rooms:** 180\n\n## Contact Information\n**Address:** 789 Avenue des Champs-Élysées, 75008 Paris\n**Phone:** +33-1-5555-0001\n**Email:** paris@anycompany.com\n\n## Check-in & Check-out\n\n### Check-in Time\nStandard check-in time is 3:00 PM. Early check-in is subject to availability and may be requested at the time of booking or by contacting the hotel directly.\n\n### Check-out Time\nStandard check-out time is 11:00 AM. Late check-out until 2:00 PM may be available for an additional fee o…

[2] score=0.8265
{'chunk': "# Crescent Quarter Elegance\n\n## Hotel Overview\nCrescent Quarter Elegance is located in New Orleans, LA, USA. Contemporary hotel perfect for business and leisure\n

## Pattern 4: Text2Cypher for flexible structured questions

Use Text2Cypher when a question expresses structured constraints over named fields and relationships. This example displays the generated query and selected database records directly.

In [9]:
hotel_schema = '''
Node properties:
Hotel {name: STRING, address: STRING, guest_rating: FLOAT, total_rooms: INTEGER}
Room {type: STRING, bed_configuration: STRING, max_occupancy: INTEGER}
Amenity {name: STRING}
Policy {name: STRING, description: STRING}
Service {name: STRING, description: STRING, cost: STRING, hours: STRING}
Relationships:
(:Hotel)-[:HAS_ROOM]->(:Room)
(:Hotel)-[:OFFERS_AMENITY]->(:Amenity)
(:Hotel)-[:HAS_POLICY]->(:Policy)
(:Hotel)-[:PROVIDES_SERVICE]->(:Service)
'''
examples = [
    "USER INPUT: Which Paris hotels have a guest rating of at least 4? CYPHER: MATCH (h:Hotel) WHERE toLower(h.address) CONTAINS 'paris' AND h.guest_rating >= 4 RETURN h.name AS hotel_name, h.guest_rating AS guest_rating ORDER BY guest_rating DESC",
    "USER INPUT: Which hotels offer a spa? CYPHER: MATCH (h:Hotel)-[:OFFERS_AMENITY]->(a:Amenity) WHERE toLower(a.name) CONTAINS 'spa' RETURN DISTINCT h.name AS hotel_name ORDER BY hotel_name",
]
text2cypher_prompt = '''
Generate one read-only Cypher 25 query for the user question.
Use only the supplied schema. Never write or delete data.
Return only the Cypher query with no markdown fence or explanation.
Schema:
{schema}
Examples:
{examples}
User question: {query_text}
'''
text2cypher_retriever = Text2CypherRetriever(
    driver=driver,
    llm=BedrockLLM(region_name=aws_region()),
    neo4j_schema=hotel_schema,
    examples=examples,
    custom_prompt=text2cypher_prompt,
)
structured_question = 'Which hotels in Cairo have a guest rating of at least 4.5 and offer a spa?'
structured_result = text2cypher_retriever.search(
    query_text=structured_question
)

print(f'Question: {structured_question}')
print(f"Generated Cypher:\n{structured_result.metadata['cypher']}\n")
print('Returned records:')
for item in structured_result.items:
    print(f'  {item.content}')
print('\nPattern fit: the database applies the field and relationship filters to each returned hotel.')

Question: Which hotels in Cairo have a guest rating of at least 4.5 and offer a spa?
Generated Cypher:
MATCH (h:Hotel)-[:OFFERS_AMENITY]->(a:Amenity)
WHERE toLower(h.address) CONTAINS 'cairo' AND h.guest_rating >= 4.5 AND toLower(a.name) CONTAINS 'spa'
RETURN DISTINCT h.name AS hotel_name, h.guest_rating AS guest_rating
ORDER BY guest_rating DESC

Returned records:
  <Record hotel_name='AnyCompany Giza Pyramids' guest_rating=4.6>
  <Record hotel_name='AnyCompany Cairo Nile View' guest_rating=4.5>

Pattern fit: the database applies the field and relationship filters to each returned hotel.


### Run Text2Cypher behind a trust boundary

In this notebook, `Text2CypherRetriever` runs in the local process. It uses the database credentials and the pinned `hotel_schema` directly. A read-only Neo4j MCP service can run the same retrieval pattern on the server. The service then owns the credentials, schema pinning, and read-only enforcement while the agent calls a governed tool. Module 4 productionizes the fixed Hybrid-Cypher pattern.

## Advanced pattern: HybridCypherRetriever

`HybridCypherRetriever` combines the exact matching of hybrid search with the graph expansion used by Vector-Cypher. Use it when a query contains an identifier and needs connected context. This advanced variation combines patterns 2 and 3.

## Choose a retrieval pattern

| Query shape | Start with | Why |
|---|---|---|
| Semantic lookup or paraphrase | VectorRetriever | Meaning matters more than exact wording |
| Exact hotel name, policy term, or identifier | HybridRetriever | Combines semantic and full-text signals |
| Semantic lookup plus connected hotel context | VectorCypherRetriever | Retrieves a chunk, then traverses relationships |
| Flexible structured filter or aggregation | Text2CypherRetriever | Generates a read-only query against the pinned schema |
| Question outside the graph | No retriever | Return an explicit empty result or abstain |

## Chunking note

This workshop uses large chunks during graph extraction so each hotel stays intact while the pipeline creates entities and relationships. A production system can keep that extraction strategy and create separate, smaller chunks for retrieval. Choose the retrieval chunk size based on the source material and query shapes. This workshop uses the prepared chunking configuration.

In [10]:
driver.close()
print('Connection closed.')

Connection closed.
